# EXP_030D — Text Ablation: Best Image + ViSoBERT + Concat + MSE
**Phase 3 | Text Backbone Ablation**
Research question: Does ViSoBERT (Vietnamese social media pretraining) improve over XLM-R for Foody reviews?
- Text model: `uitnlp/visobert` | Image model: Best from Phase 2 (set below)
- Fusion: Concatenation + MLP | Loss: MSE | Seed: 42 | AMP: enabled
> ⚠️ Prerequisite: Phase 2 must be completed. Set BEST_IMAGE_MODEL in STEP 4.

### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### STEP 2: Clone source code and install dependencies

In [ ]:
!git clone https://github.com/lechihoang/SE365.git
%cd SE365
!pip install -r requirements.txt -q

Cloning into 'SE365'...
remote: Enumerating objects: 13344, done.
remote: Counting objects: 100% (215/215), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 13344 (delta 152), reused 156 (delta 96), pack-reused 13129 (from 1)
Receiving objects: 100% (13344/13344), 873.19 MiB | 20.12 MiB/s, done.
Resolving deltas: 100% (390/390), done.
/content/SE365


### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!cp /content/drive/MyDrive/SE365/data.zip ./data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

total 1344
drwxr-xr-x  4 root root    4096 Jun 16 09:21 .
drwxr-xr-x 12 root root    4096 Jun 23 11:06 ..
drwxr-xr-x  2 root root 1359872 Jun 16 09:59 image
drwxr-xr-x  2 root root    4096 Jun 16 09:21 text


### STEP 4: Configure paths — ✏️ Set Phase 2 winner here

In [ ]:
import os
DRIVE_ROOT = '/content/drive/MyDrive/SE365'  # ✏️ Change if needed
EXP_ID = 'EXP_030D_bestimage_visobert_concat_mse'

# ✏️ SET based on Phase 2 results (same as EXP_030B)
BEST_IMAGE_MODEL  = 'swin_base_patch4_window7_224'
BEST_IMAGE_EXP_ID = 'EXP_020B_swinb_xlmr_concat_mse'

DRIVE_EXP_PATH = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
os.makedirs(DRIVE_EXP_PATH, exist_ok=True)
print(f'Artifacts will be saved to: {DRIVE_EXP_PATH}')
print(f'Using image backbone  : {BEST_IMAGE_MODEL}')
print(f'Loading image weights : {BEST_IMAGE_EXP_ID}')

Artifacts will be saved to: /content/drive/MyDrive/SE365/experiments/EXP_030D_bestimage_visobert_concat_mse
Using image backbone  : swin_base_patch4_window7_224
Loading image weights : EXP_020B_swinb_xlmr_concat_mse


### STEP 5: Load image weights from Phase 2 winner


In [ ]:
import os, shutil
os.makedirs('./checkpoints', exist_ok=True)
shutil.copy(f'{DRIVE_ROOT}/experiments/{BEST_IMAGE_EXP_ID}/best_model_train_image.pth', './checkpoints/best_model_train_image.pth')
print(f'Loaded image weights from {BEST_IMAGE_EXP_ID}')


Loaded image weights from EXP_020B_swinb_xlmr_concat_mse


### STEP 6: Pre-train Text Branch
Fine-tune the text model before fusion to align its features with the score prediction task.

In [ ]:
!python main.py \
  --mode train_text \
  --text_model_name uitnlp/visobert \
  --epochs 20 \
  --batch_size 16 \
  --lr 1e-5 \
  --loss_fn mse \
  --seed 42 \
  --use_amp \
  --exp_dir ./experiments

!cp ./checkpoints/best_model_train_text.pth ./experiments/$EXP_ID/

====== MODE: TRAIN_TEXT ======
Using device: cuda
Seed: 42 | Experiment: EXP_000
config.json: 100% 644/644 [00:00<00:00, 3.70MB/s]
sentencepiece.bpe.model: 100% 471k/471k [00:01<00:00, 243kB/s]
Loaded timm processor for convnext_base_in22k
pytorch_model.bin: 100% 390M/390M [00:03<00:00, 103MB/s] 
Loading weights: 100% 197/197 [00:00<00:00, 16688.44it/s]
[transformers] XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

### STEP 7: Train Fusion Model


In [ ]:
!python main.py \
  --mode train_fusion \
  --text_model_name uitnlp/visobert \
  --image_model_name {BEST_IMAGE_MODEL} \
  --epochs 15 \
  --batch_size 16 \
  --lr 1e-5 \
  --grad_accum_steps 2 \
  --patience 5 \
  --loss_fn mse \
  --unfreeze_text_layers 1 \
  --unfreeze_image_layers 1 \
  --seed 42 \
  --use_amp \
  --exp_id EXP_030D_bestimage_visobert_concat_mse \
  --exp_dir ./experiments

====== MODE: TRAIN_FUSION ======
Using device: cuda
Seed: 42 | Experiment: EXP_030D_bestimage_visobert_concat_mse
Loaded timm processor for swin_base_patch4_window7_224
Loading weights: 100% 197/197 [00:00<00:00, 17584.50it/s]
[transformers] XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
model.safetensors: 100% 353M/353M [00:02<00:00, 130MB/s]
/content/SE365/Trainer.py

### STEP 8: Save to Drive + print metrics


In [ ]:
import json
!cp -r ./experiments/$EXP_ID/* $DRIVE_EXP_PATH/

with open(f'./experiments/{EXP_ID}/metrics.json') as f:
    m = json.load(f)

print(f'\n=== {EXP_ID} Results ===')
print(f"Loss (val)   : {m['loss']:.4f}")
print()
print("             MAE      RMSE      R2")
print(f"  food     : {m['mae_food']:.4f}   {m['rmse_food']:.4f}   {m['r2_food']:.4f}")
print(f"  price    : {m['mae_price']:.4f}   {m['rmse_price']:.4f}   {m['r2_price']:.4f}")
print(f"  atmos    : {m['mae_atmos']:.4f}   {m['rmse_atmos']:.4f}   {m['r2_atmos']:.4f}")
print(f"  service  : {m['mae_service']:.4f}   {m['rmse_service']:.4f}   {m['r2_service']:.4f}")
print(f"  overall  : {m['mae_overall']:.4f}   {m['rmse_overall']:.4f}   {m['r2_overall']:.4f}")
print()
print(f"  mean_mae   : {m['mean_mae']:.4f}")
print(f"  aspect_mae : {m['aspect_mae']:.4f}")
print(f"  overall_mae: {m['overall_mae']:.4f}")


=== EXP_030D_bestimage_visobert_concat_mse Results ===
Loss (val)   : 2.8100

             MAE      RMSE      R2
  food     : 1.2576   1.7451   0.4214
  price    : 1.2736   1.7649   0.3026
  atmos    : 1.2459   1.6386   0.3081
  service  : 1.2945   1.7339   0.4137
  overall  : 1.0923   1.4843   0.4589

  mean_mae   : 1.2328
  aspect_mae : 1.2679
  overall_mae: 1.0923
